# digits-28 · Treino, avaliação e exportação da CNN

Notebook de apoio ao reconhecedor de dígitos manuscritos que roda **inteiramente no navegador**.

Este notebook cobre as Partes 1 a 3 da atividade:

| Parte | O que acontece aqui |
|---|---|
| **1 · Treinar** | CNN treinada no MNIST completo (60.000 treino / 10.000 teste) |
| **2 · Avaliar** | Acurácia, matriz de confusão 10x10, par mais confundido e amostras de erro |
| **3 · Exportar** | Conversão para TensorFlow.js (`modelo_web/`) + `relatorio.json` |

Ao final, uma célula gera **`entrega.zip`** com tudo que a página web precisa.

> **Runtime:** `Ambiente de execução → Alterar o tipo de ambiente de execução → GPU (T4)`.
> O treino leva cerca de 3 a 6 minutos na T4.

In [ ]:
# ============================================================
# 0 - Ambiente
# ============================================================
import sys, platform, json, os, io, base64, zipfile, datetime
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

SEED = 42
keras.utils.set_random_seed(SEED)

gpus = tf.config.list_physical_devices('GPU')
print('Python      :', platform.python_version())
print('TensorFlow  :', tf.__version__)
print('Keras       :', keras.__version__)
print('GPU         :', gpus[0].name if gpus else 'NENHUMA (o treino vai rodar em CPU e demorar mais)')
if gpus:
    print('              ->', tf.config.experimental.get_device_details(gpus[0]).get('device_name', 'desconhecida'))

---
## Parte 1 · Dados

MNIST completo, sem subamostragem: 60.000 imagens de treino e 10.000 de teste, 28x28 em escala de cinza.

Duas decisões aqui:

1. **Normalização para `[0, 1]`** dividindo por 255. A mesma escala é aplicada no navegador (`.div(255)`), então treino e inferência veem exatamente a mesma faixa de valores.
2. **Separação de validação a partir do treino (10%)**, e não do teste. O `EarlyStopping` precisa de um conjunto para decidir quando parar — usar o teste para isso vazaria informação e inflaria a acurácia reportada. Os 10.000 exemplos de teste ficam intocados até a Parte 2.

In [ ]:
# ============================================================
# 1 - Carregamento e preparo dos dados
# ============================================================
(x_tr_full, y_tr_full), (x_te, y_te) = keras.datasets.mnist.load_data()

x_tr_full = (x_tr_full / 255.0).astype('float32')[..., None]
x_te      = (x_te / 255.0).astype('float32')[..., None]
y_tr_full = y_tr_full.astype('int64')
y_te      = y_te.astype('int64')

# separacao treino / validacao (o teste permanece intocado)
rng = np.random.default_rng(SEED)
perm    = rng.permutation(len(x_tr_full))
n_val   = 6000
idx_val, idx_tr = perm[:n_val], perm[n_val:]

x_tr, y_tr = x_tr_full[idx_tr], y_tr_full[idx_tr]
x_val, y_val = x_tr_full[idx_val], y_tr_full[idx_val]

print(f'treino    : {x_tr.shape}  rotulos {y_tr.shape}')
print(f'validacao : {x_val.shape}  rotulos {y_val.shape}')
print(f'teste     : {x_te.shape}  rotulos {y_te.shape}')
print(f'faixa de valores: [{x_tr.min():.1f}, {x_tr.max():.1f}]')
print('distribuicao das classes no treino:', np.bincount(y_tr))

In [ ]:
# ============================================================
# 1b - Amostra visual do dataset
# ============================================================
fig, axes = plt.subplots(2, 10, figsize=(14, 3.2))
for d in range(10):
    ids = np.where(y_tr == d)[0][:2]
    for r in range(2):
        ax = axes[r, d]
        ax.imshow(x_tr[ids[r], :, :, 0], cmap='gray')
        ax.axis('off')
        if r == 0:
            ax.set_title(str(d), fontsize=12)
fig.suptitle('MNIST — duas amostras de cada classe', y=1.04)
plt.tight_layout(); plt.show()

---
## Parte 1 · Arquitetura e por que ela é assim

A rede é uma CNN em **três blocos convolucionais** seguidos de um classificador denso curto.

```
Entrada 28x28x1
│
├─ Bloco 1   Conv 32 (3x3) → Conv 32 (3x3) → BatchNorm → MaxPool 2x2 → Dropout 0.25    28x28 → 14x14
├─ Bloco 2   Conv 64 (3x3) → Conv 64 (3x3) → BatchNorm → MaxPool 2x2 → Dropout 0.25    14x14 →  7x7
├─ Bloco 3   Conv 128 (3x3)                → BatchNorm → MaxPool 2x2 → Dropout 0.25     7x7  →  3x3
│
└─ Flatten (1152) → Dense 128 → BatchNorm → Dropout 0.4 → Dense 10 (softmax)
```

**Por que dois `Conv` seguidos antes de cada pooling.** Duas convoluções 3x3 empilhadas enxergam a mesma região que uma 5x5, mas com menos parâmetros e uma não-linearidade a mais no meio. Traço manuscrito é feito de bordas e curvas curtas — vale mais profundidade barata do que filtros largos.

**Por que o número de filtros dobra a cada bloco.** Cada pooling corta a resolução pela metade, então a informação espacial diminui; dobrar os canais compensa isso deslocando a representação de "onde está a borda" para "que tipo de traço é". É o padrão de funil das CNNs clássicas.

**Por que `BatchNormalization`.** Normalizar as ativações dentro do bloco estabiliza o gradiente e permite uma taxa de aprendizado maior sem divergir. Na prática, é o que faz a rede chegar em ~99% em poucas épocas em vez de dezenas.

**Por que `Dropout` crescente (0.25 → 0.4).** As camadas convolucionais já são regularizadas pelo compartilhamento de pesos; a densa de 128 unidades é a que mais decora. O dropout mais forte fica onde está o risco de overfitting.

**Por que só 3x3x128 antes do `Flatten`.** Chegar em um mapa espacial pequeno antes de achatar mantém a camada densa com 147k parâmetros em vez de 400k+. O modelo inteiro fica em torno de **1,1 MB**, e esse arquivo é baixado pelo navegador de quem abre a página — tamanho aqui é experiência do usuário, não só elegância.

**Aumento de dados.** Rotação ±14°, translação ±10% e zoom ±10%. Isso não é enfeite: o MNIST é composto de dígitos já centralizados e normalizados, enquanto o usuário desenha torto, deslocado e com espessura variável. O aumento aproxima o treino da distribuição real de uso. As camadas de aumento ficam **fora do modelo**, aplicadas no `tf.data` — assim o modelo exportado contém só o que precisa rodar na inferência.

In [ ]:
# ============================================================
# 1c - Definicao da arquitetura
# ============================================================
def construir_modelo():
    return keras.Sequential([
        keras.Input(shape=(28, 28, 1), name='entrada'),

        # bloco 1 -------------------------------------------------- 28x28 -> 14x14
        layers.Conv2D(32, 3, padding='same', activation='relu', name='conv1a'),
        layers.Conv2D(32, 3, padding='same', activation='relu', name='conv1b'),
        layers.BatchNormalization(name='bn1'),
        layers.MaxPooling2D(2, name='pool1'),
        layers.Dropout(0.25, name='drop1'),

        # bloco 2 -------------------------------------------------- 14x14 -> 7x7
        layers.Conv2D(64, 3, padding='same', activation='relu', name='conv2a'),
        layers.Conv2D(64, 3, padding='same', activation='relu', name='conv2b'),
        layers.BatchNormalization(name='bn2'),
        layers.MaxPooling2D(2, name='pool2'),
        layers.Dropout(0.25, name='drop2'),

        # bloco 3 -------------------------------------------------- 7x7 -> 3x3
        layers.Conv2D(128, 3, padding='same', activation='relu', name='conv3a'),
        layers.BatchNormalization(name='bn3'),
        layers.MaxPooling2D(2, name='pool3'),
        layers.Dropout(0.25, name='drop3'),

        # classificador -------------------------------------------------------
        layers.Flatten(name='flatten'),
        layers.Dense(128, activation='relu', name='densa'),
        layers.BatchNormalization(name='bn4'),
        layers.Dropout(0.4, name='drop4'),
        layers.Dense(10, activation='softmax', name='saida'),
    ], name='digits28_cnn')

modelo = construir_modelo()
modelo.summary()

In [ ]:
# ============================================================
# 1d - Pipeline de dados com aumento (fora do modelo)
# ============================================================
AUTOTUNE   = tf.data.AUTOTUNE
BATCH_SIZE = 128

aumento = keras.Sequential([
    layers.RandomRotation(0.04, fill_mode='constant', fill_value=0.0),      # +/- ~14 graus
    layers.RandomTranslation(0.10, 0.10, fill_mode='constant', fill_value=0.0),
    layers.RandomZoom(0.10, fill_mode='constant', fill_value=0.0),
], name='aumento')

ds_treino = (tf.data.Dataset.from_tensor_slices((x_tr, y_tr))
             .shuffle(len(x_tr), seed=SEED)
             .batch(BATCH_SIZE)
             .map(lambda a, b: (aumento(a, training=True), b), num_parallel_calls=AUTOTUNE)
             .prefetch(AUTOTUNE))

ds_val = (tf.data.Dataset.from_tensor_slices((x_val, y_val))
          .batch(BATCH_SIZE)
          .prefetch(AUTOTUNE))

# conferencia visual do aumento
lote = next(iter(ds_treino))[0].numpy()
fig, axes = plt.subplots(1, 10, figsize=(14, 1.7))
for i, ax in enumerate(axes):
    ax.imshow(lote[i, :, :, 0], cmap='gray'); ax.axis('off')
fig.suptitle('Exemplos após aumento de dados', y=1.15)
plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# 1e - Compilacao, callbacks e treino
# ============================================================
modelo.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

parada = keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=6,
    restore_best_weights=True,   # exigido pelo enunciado
    verbose=1,
)

# reduz a taxa de aprendizado quando a validacao estaciona:
# deixa o modelo refinar os ultimos decimos de acuracia antes do EarlyStopping agir
reduz_lr = keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', factor=0.5, patience=3, min_lr=1e-5, verbose=1,
)

historico = modelo.fit(
    ds_treino,
    validation_data=ds_val,
    epochs=40,
    callbacks=[parada, reduz_lr],
    verbose=2,
)

hist = {k: [float(v) for v in vs] for k, vs in historico.history.items()}
epoca_melhor = int(np.argmin(hist['val_loss'])) + 1
print(f'\nepocas executadas : {len(hist["loss"])}')
print(f'melhor epoca      : {epoca_melhor} (val_loss = {min(hist["val_loss"]):.5f})')

In [ ]:
# ============================================================
# 1f - Curvas de aprendizado
# ============================================================
eixo = range(1, len(hist['loss']) + 1)
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 4))

a1.plot(eixo, hist['loss'], label='treino')
a1.plot(eixo, hist['val_loss'], label='validação')
a1.axvline(epoca_melhor, ls='--', c='gray', lw=1)
a1.set_title('Perda'); a1.set_xlabel('época'); a1.legend(); a1.grid(alpha=.25)

a2.plot(eixo, hist['accuracy'], label='treino')
a2.plot(eixo, hist['val_accuracy'], label='validação')
a2.axvline(epoca_melhor, ls='--', c='gray', lw=1)
a2.set_title('Acurácia'); a2.set_xlabel('época'); a2.legend(); a2.grid(alpha=.25)

fig.suptitle(f'Curvas de aprendizado — melhor época: {epoca_melhor} (linha tracejada)')
plt.tight_layout(); plt.show()

---
## Parte 2 · Avaliar antes de exportar

Nada é exportado antes desta seção rodar. O que é medido:

- **acurácia no conjunto de teste** (os 10.000 exemplos nunca vistos)
- **matriz de confusão 10x10**
- **os dois dígitos que o modelo mais confunde entre si**
- **imagens que ele errou**, com a previsão e o rótulo verdadeiro

A acurácia sozinha esconde o formato do erro: 99,2% pode significar erros espalhados por igual ou uma classe específica quebrada. A matriz é quem mostra isso.

In [ ]:
# ============================================================
# 2a - Acuracia no conjunto de teste
# ============================================================
perda_teste, acuracia_teste = modelo.evaluate(x_te, y_te, batch_size=256, verbose=0)

probs = modelo.predict(x_te, batch_size=256, verbose=0)
y_pred = probs.argmax(axis=1)
conf_pred = probs.max(axis=1)

print(f'acurácia no teste : {acuracia_teste*100:.2f}%')
print(f'perda no teste    : {perda_teste:.5f}')
print(f'erros             : {int((y_pred != y_te).sum())} de {len(y_te)}')

In [ ]:
# ============================================================
# 2b - Matriz de confusao 10x10
# ============================================================
mc = np.zeros((10, 10), dtype=int)
for real, prev in zip(y_te, y_pred):
    mc[real, prev] += 1

fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(mc, cmap='Blues', norm=plt.matplotlib.colors.LogNorm(vmin=1, vmax=mc.max()))
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel('previsto pelo modelo'); ax.set_ylabel('rótulo verdadeiro')
ax.set_title('Matriz de confusão — conjunto de teste (10.000 imagens)')

for i in range(10):
    for j in range(10):
        v = mc[i, j]
        if v == 0:
            continue
        ax.text(j, i, str(v), ha='center', va='center', fontsize=8,
                color='white' if (i == j or v > mc.max() * 0.02) else '#1f3b63',
                fontweight='bold' if i == j else 'normal')

plt.colorbar(im, ax=ax, label='contagem (escala log)')
plt.tight_layout(); plt.show()

# metricas por classe, derivadas da matriz
por_classe = []
for d in range(10):
    vp = mc[d, d]
    fn = mc[d, :].sum() - vp
    fp = mc[:, d].sum() - vp
    prec = vp / (vp + fp) if (vp + fp) else 0.0
    rev  = vp / (vp + fn) if (vp + fn) else 0.0
    f1   = 2 * prec * rev / (prec + rev) if (prec + rev) else 0.0
    por_classe.append({'digito': d, 'precisao': float(prec), 'revocacao': float(rev),
                       'f1': float(f1), 'suporte': int(mc[d, :].sum()), 'acertos': int(vp)})

print(f'{"dígito":>7} {"precisão":>10} {"revocação":>11} {"F1":>8} {"suporte":>9}')
for m in por_classe:
    print(f'{m["digito"]:>7} {m["precisao"]*100:>9.2f}% {m["revocacao"]*100:>10.2f}% {m["f1"]*100:>7.2f}% {m["suporte"]:>9}')

In [ ]:
# ============================================================
# 2c - Os dois digitos que o modelo mais confunde ENTRE SI
# ============================================================
# confusao direcional: real -> previsto
direcionais = [{'de': int(i), 'para': int(j), 'n': int(mc[i, j])}
               for i in range(10) for j in range(10) if i != j and mc[i, j] > 0]
direcionais.sort(key=lambda d: -d['n'])

# confusao MUTUA: soma dos dois sentidos, que e o que o enunciado pede
pares = [{'a': i, 'b': j, 'total': int(mc[i, j] + mc[j, i]),
          'a_para_b': int(mc[i, j]), 'b_para_a': int(mc[j, i])}
         for i in range(10) for j in range(i + 1, 10)]
pares.sort(key=lambda p: -p['total'])
par_top = pares[0]

print('Par mais confundido entre si:')
print(f"  {par_top['a']} <-> {par_top['b']}  —  {par_top['total']} erros no total")
print(f"    {par_top['a']} classificado como {par_top['b']}: {par_top['a_para_b']}")
print(f"    {par_top['b']} classificado como {par_top['a']}: {par_top['b_para_a']}")

print('\n5 confusões mútuas mais frequentes:')
for p in pares[:5]:
    print(f"  {p['a']} <-> {p['b']:<2} {p['total']:>3} erros")

print('\n5 confusões direcionais mais frequentes (real -> previsto):')
for d in direcionais[:5]:
    print(f"  {d['de']} -> {d['para']:<2} {d['n']:>3} erros")

In [ ]:
# ============================================================
# 2d - Imagens que o modelo errou
# ============================================================
idx_erros = np.where(y_pred != y_te)[0]
# ordena pelos erros mais "confiantes": onde errou e ainda assim tinha certeza
idx_erros = idx_erros[np.argsort(-conf_pred[idx_erros])]

N_MOSTRAR = 8
selecao = idx_erros[:N_MOSTRAR]

fig, axes = plt.subplots(1, N_MOSTRAR, figsize=(2 * N_MOSTRAR, 2.9))
for ax, i in zip(axes, selecao):
    ax.imshow(x_te[i, :, :, 0], cmap='gray')
    ax.set_title(f'previu {y_pred[i]}\nera {y_te[i]}  ({conf_pred[i]*100:.0f}%)',
                 fontsize=10, color='#b91c1c')
    ax.axis('off')
fig.suptitle('Erros mais confiantes do modelo no conjunto de teste', y=1.06)
plt.tight_layout(); plt.show()

# codifica as amostras de erro em PNG base64 para o relatorio da pagina web
from PIL import Image

def png_base64(arr28):
    img = Image.fromarray((arr28 * 255).clip(0, 255).astype('uint8'), mode='L')
    buf = io.BytesIO()
    img.save(buf, format='PNG')
    return 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode('ascii')

amostras_erro = []
for i in selecao:
    ordem = np.argsort(-probs[i])
    amostras_erro.append({
        'indice': int(i),
        'previsto': int(y_pred[i]),
        'real': int(y_te[i]),
        'confianca': float(conf_pred[i]),
        'conf_classe_real': float(probs[i, y_te[i]]),
        'segundo': int(ordem[1]),
        'png': png_base64(x_te[i, :, :, 0]),
    })

print(f'{len(idx_erros)} erros no total; {len(amostras_erro)} exportados para o relatorio.')

---
## Parte 3 · Exportar

Duas rotas, nesta ordem:

1. **`tensorflowjs` oficial** — `tfjs.converters.save_keras_model(...)`. É o caminho do enunciado e o primeiro a ser tentado.
2. **Exportador embutido (plano B)** — o pacote `tensorflowjs` costuma exigir uma versão de TensorFlow diferente da que o Colab traz, e a instalação quebra o runtime com frequência. A função `exportar_tfjs` abaixo escreve o mesmo formato `layers-model` (um `model.json` com a topologia e um `.bin` com os pesos em float32) direto do Keras 3, sem dependência extra.

A célula tenta a rota 1 e cai para a rota 2 automaticamente. O resultado é equivalente do ponto de vista do `tf.loadLayersModel()` no navegador — e a célula seguinte confere se o arquivo gerado bate com o modelo treinado, tensor por tensor.

In [ ]:
# ============================================================
# 3a - Salva o modelo Keras
# ============================================================
modelo.save('modelo.keras')
print('modelo.keras salvo:', round(os.path.getsize('modelo.keras') / 1e6, 2), 'MB')

In [ ]:
# ============================================================
# 3b - Exportador para o formato layers-model do TensorFlow.js
#      (plano B, sem depender do pacote tensorflowjs)
# ============================================================
CHAVES_FORA_DA_CAMADA = ('registered_name', 'module', 'build_config', 'shared_object_id')
CHAVES_FORA_DO_CONFIG = ('synchronized', 'autocast', 'seed_generator', 'sparse', 'ragged', 'optional')

def _limpar(no):
    # remove os campos que o Keras 3 emite e o parser do TF.js nao conhece
    if isinstance(no, list):
        return [_limpar(x) for x in no]
    if not isinstance(no, dict):
        return no

    no = {k: v for k, v in no.items() if k not in CHAVES_FORA_DA_CAMADA}

    cfg = no.get('config')
    if isinstance(cfg, dict):
        # Keras 3 usa batch_shape; o TF.js espera batch_input_shape
        if 'batch_shape' in cfg and 'batch_input_shape' not in cfg:
            cfg['batch_input_shape'] = cfg.pop('batch_shape')
        # Keras 3 usa um dict de DTypePolicy; o TF.js espera a string
        if isinstance(cfg.get('dtype'), dict):
            cfg['dtype'] = cfg['dtype'].get('config', {}).get('name', 'float32')
        for k in CHAVES_FORA_DO_CONFIG:
            cfg.pop(k, None)
        no['config'] = _limpar(cfg)

    for k, v in list(no.items()):
        if k != 'config' and isinstance(v, (dict, list)):
            no[k] = _limpar(v)
    return no

def exportar_tfjs(modelo, pasta='modelo_web', nome_bin='group1-shard1of1.bin'):
    os.makedirs(pasta, exist_ok=True)
    bruto = json.loads(modelo.to_json())
    camadas = bruto.get('config', {}).get('layers', [])

    topologia = {
        'keras_version': '2.15.0',   # versao de config que o parser do TF.js entende
        'backend': 'tensorflow',
        'model_config': {
            'class_name': 'Sequential',
            'config': {
                'name': bruto.get('config', {}).get('name', 'sequential'),
                'layers': [_limpar(c) for c in camadas],
            },
        },
    }

    manifesto, blocos = [], []
    for camada in modelo.layers:
        for peso in camada.weights:
            arr = np.asarray(peso.numpy(), dtype=np.float32)
            # o TF.js procura cada peso por "<nome da camada>/<nome do peso>"
            manifesto.append({
                'name': f'{camada.name}/{peso.name}',
                'shape': list(arr.shape),
                'dtype': 'float32',
            })
            blocos.append(arr.tobytes())

    with open(os.path.join(pasta, nome_bin), 'wb') as f:
        for b in blocos:
            f.write(b)

    with open(os.path.join(pasta, 'model.json'), 'w', encoding='utf-8') as f:
        json.dump({
            'format': 'layers-model',
            'generatedBy': f'keras v{keras.__version__}',
            'convertedBy': 'digits-28 exportador embutido',
            'modelTopology': topologia,
            'weightsManifest': [{'paths': [nome_bin], 'weights': manifesto}],
        }, f, indent=2)

    return manifesto

print('exportador definido.')

In [ ]:
# ============================================================
# 3c - Conversao: tenta o tensorflowjs oficial, cai para o plano B
# ============================================================
PASTA_WEB = 'modelo_web'
rota = None

try:
    import tensorflowjs as tfjs
    tfjs.converters.save_keras_model(modelo, PASTA_WEB)
    rota = f'tensorflowjs {tfjs.__version__} (rota oficial)'
except Exception as erro:
    print('rota oficial indisponivel:', type(erro).__name__, '-', str(erro)[:160])
    print('usando o exportador embutido...')
    exportar_tfjs(modelo, PASTA_WEB)
    rota = 'exportador embutido (digits-28)'

print()
print('rota usada:', rota)
for nome in sorted(os.listdir(PASTA_WEB)):
    kb = os.path.getsize(os.path.join(PASTA_WEB, nome)) / 1024
    print(f'  {nome:<28} {kb:>9.1f} KB')

tamanho_web_kb = sum(os.path.getsize(os.path.join(PASTA_WEB, n)) for n in os.listdir(PASTA_WEB)) / 1024
print(f'total baixado pelo navegador: {tamanho_web_kb:.0f} KB')

In [ ]:
# ============================================================
# 3d - Conferencia: o modelo exportado ainda e o mesmo modelo?
# ============================================================
with open(os.path.join(PASTA_WEB, 'model.json'), encoding='utf-8') as f:
    mj = json.load(f)

esperado = {f'{c.name}/{p.name}': tuple(p.shape) for c in modelo.layers for p in c.weights}
declarado = {w['name']: tuple(w['shape'])
             for grupo in mj['weightsManifest'] for w in grupo['weights']}

faltando = sorted(set(esperado) - set(declarado))
divergente = sorted(k for k in set(esperado) & set(declarado) if esperado[k] != declarado[k])

bytes_esperados = sum(int(np.prod(s)) * 4 for s in declarado.values())
bytes_reais = sum(os.path.getsize(os.path.join(PASTA_WEB, p))
                  for grupo in mj['weightsManifest'] for p in grupo['paths'])

print(f'tensores de peso  : {len(declarado)} declarados / {len(esperado)} no modelo')
print(f'pesos faltando    : {faltando if faltando else "nenhum"}')
print(f'formas divergentes: {divergente if divergente else "nenhuma"}')
print(f'bytes no .bin     : {bytes_reais} (esperado {bytes_esperados})')

ok = (not faltando) and (not divergente) and (bytes_reais == bytes_esperados)
print()
print('CONFERIDO - modelo_web/ esta consistente com o modelo treinado.' if ok
      else 'ATENCAO - inconsistencia detectada, nao publique este modelo.')
assert ok, 'exportacao inconsistente'

---
## Relatório em JSON

A página web não traz números escritos à mão no HTML. Ela lê o `relatorio.json` gerado abaixo e monta a seção de avaliação a partir dele — acurácia, matriz de confusão, curvas de treino, par mais confundido e as amostras de erro.

Isso tem uma consequência boa: **retreinar o modelo atualiza o relatório publicado automaticamente**. Não existe a possibilidade de a página exibir a acurácia de um modelo diferente do que está carregado nela.

In [ ]:
# ============================================================
# 4 - Monta o relatorio.json consumido pela pagina web
# ============================================================
def forma_saida(camada):
    try:
        return [None if d is None else int(d) for d in camada.output.shape]
    except Exception:
        return None

arquitetura = [{
    'nome': c.name,
    'tipo': c.__class__.__name__,
    'saida': forma_saida(c),
    'params': int(c.count_params()),
} for c in modelo.layers]

relatorio = {
    'gerado_em': datetime.datetime.now().astimezone().isoformat(timespec='seconds'),
    'ambiente': {
        'tensorflow': tf.__version__,
        'keras': keras.__version__,
        'python': platform.python_version(),
        'gpu': (tf.config.experimental.get_device_details(gpus[0]).get('device_name', gpus[0].name)
                if gpus else 'CPU'),
        'rota_exportacao': rota,
    },
    'dados': {
        'treino': int(len(x_tr)),
        'validacao': int(len(x_val)),
        'teste': int(len(x_te)),
        'aumento': ['rotacao +/-14 graus', 'translacao +/-10%', 'zoom +/-10%'],
    },
    'treino': {
        'epocas_executadas': int(len(hist['loss'])),
        'epoca_melhor': int(epoca_melhor),
        'batch_size': int(BATCH_SIZE),
        'otimizador': 'Adam',
        'lr_inicial': 1e-3,
        'perda': 'sparse_categorical_crossentropy',
        'early_stopping': {'monitor': 'val_loss', 'patience': 6, 'restore_best_weights': True},
        'historico': hist,
    },
    'teste': {
        'acuracia': float(acuracia_teste),
        'perda': float(perda_teste),
        'n': int(len(y_te)),
        'erros': int((y_pred != y_te).sum()),
    },
    'matriz_confusao': mc.tolist(),
    'por_classe': por_classe,
    'confusoes_mutuas': pares[:8],
    'confusoes_direcionais': direcionais[:8],
    'par_mais_confundido': par_top,
    'amostras_erro': amostras_erro,
    'arquitetura': arquitetura,
    'params_total': int(modelo.count_params()),
    'tamanho_web_kb': round(tamanho_web_kb, 1),
    'camada_ativacoes': 'conv1a',   # camada que a pagina visualiza ao vivo
}

with open('relatorio.json', 'w', encoding='utf-8') as f:
    json.dump(relatorio, f, ensure_ascii=False)

print('relatorio.json escrito:', round(os.path.getsize('relatorio.json') / 1024, 1), 'KB')
print()
print(f"  acuracia no teste     : {relatorio['teste']['acuracia']*100:.2f}%")
print(f"  erros                 : {relatorio['teste']['erros']} / {relatorio['teste']['n']}")
print(f"  par mais confundido   : {par_top['a']} <-> {par_top['b']} ({par_top['total']} erros)")
print(f"  parametros            : {relatorio['params_total']:,}")
print(f"  download do navegador : {relatorio['tamanho_web_kb']:.0f} KB")

In [ ]:
# ============================================================
# 5 - Empacota tudo e baixa
# ============================================================
NOME_ZIP = 'entrega.zip'

with zipfile.ZipFile(NOME_ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for raiz, _, arquivos in os.walk(PASTA_WEB):
        for a in arquivos:
            caminho = os.path.join(raiz, a)
            z.write(caminho, os.path.relpath(caminho, '.'))
    z.write('relatorio.json')
    z.write('modelo.keras')

print(f'{NOME_ZIP} ({os.path.getsize(NOME_ZIP)/1e6:.2f} MB) contem:')
with zipfile.ZipFile(NOME_ZIP) as z:
    for n in z.namelist():
        print('  ', n)

try:
    from google.colab import files
    files.download(NOME_ZIP)
except Exception:
    print()
    print('(fora do Colab: pegue o entrega.zip na pasta de trabalho)')

---
## Partes 4 e 5 · O que fazer com o `entrega.zip`

O zip baixado contém três coisas:

| Arquivo | Para quê |
|---|---|
| `modelo_web/` | o modelo que o navegador carrega com `tf.loadLayersModel()` |
| `relatorio.json` | os números que a página exibe na seção de avaliação |
| `modelo.keras` | o modelo original, para retreinar ou converter de novo sem repetir o treino |

Descompacte na raiz do repositório, sobrescrevendo o que já existe:

```
digits-28/
├── index.html          <- a página (já está no repositório)
├── modelo_web/         <- vem do zip
│   ├── model.json
│   └── group1-shard1of1.bin
├── relatorio.json      <- vem do zip
└── modelo.keras        <- vem do zip
```

Depois, para testar localmente antes de publicar (o navegador bloqueia `fetch` em `file://`, então é preciso um servidor):

```bash
python -m http.server 8000
# abra http://localhost:8000
```

E para publicar: `git add -A && git commit && git push`, com o **GitHub Pages** apontado para a branch. A página não precisa de servidor nenhum depois de carregada — o modelo roda inteiro no navegador.